# Tgt 02: Deploy Dashboards

Runs on the **target** workspace. Reads transformed dashboards and metadata from the import volume and creates dashboards in this workspace using the default WorkspaceClient (OAuth). No cross-workspace calls or IP detection.

## Install dependencies

Install databricks-sdk and other required packages for deployment.

In [ ]:
%pip install -U databricks-sdk pandas pyyaml "numpy<2" --quiet
dbutils.library.restartPython()

## Setup helpers path

Add the helpers module to sys.path so we can import build_deployment_packages and deploy_via_sdk.

In [ ]:
import sys
import os

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    if not notebook_path.startswith('/Workspace'):
        notebook_path = '/Workspace' + notebook_path
    notebook_dir = os.path.dirname(notebook_path)
    bundle_parent = os.path.dirname(notebook_dir)
    if bundle_parent not in sys.path:
        sys.path.insert(0, bundle_parent)
except Exception:
    sys.path.insert(0, os.path.abspath('../'))

from helpers.deployment_package import build_deployment_packages, load_permissions_from_csv, load_schedules_from_csv
from helpers.sdk_deployer import deploy_via_sdk, resolve_warehouse
from databricks.sdk import WorkspaceClient
import time

print("Helpers loaded.")

## Load configuration from widgets

Read volume path (import volume), paths, warehouse, and options. All parameters are passed by the target job.

In [ ]:
volume_base = dbutils.widgets.get('volume_base')
exported_path_rel = dbutils.widgets.get('exported_path')
transformed_path_rel = dbutils.widgets.get('transformed_path')
target_parent_path = dbutils.widgets.get('target_parent_path')
warehouse_id = dbutils.widgets.get('warehouse_id') or None
warehouse_name = dbutils.widgets.get('warehouse_name') or None
apply_permissions = dbutils.widgets.get('apply_permissions').lower() == 'true'
apply_schedules = dbutils.widgets.get('apply_schedules').lower() == 'true'
embed_credentials = dbutils.widgets.get('embed_credentials').lower() == 'true'
skip_duplicate_check = dbutils.widgets.get('skip_duplicate_check').lower() == 'true'
dry_run_mode = dbutils.widgets.get('dry_run_mode').lower() == 'true'

exported_path = f"{volume_base}/{exported_path_rel}"
transformed_path = f"{volume_base}/{transformed_path_rel}"

print(f"Volume base: {volume_base}")
print(f"Transformed: {transformed_path}")
print(f"Target path: {target_parent_path}")
print(f"Dry run: {dry_run_mode}")

## Load permissions and schedules from volume

Load permissions and schedules CSVs from the exported path (written by source export step).

In [ ]:
try:
    permissions_map = load_permissions_from_csv(exported_path) if apply_permissions else {}
    print(f"Loaded permissions for {len(permissions_map)} dashboard(s)")
except FileNotFoundError:
    permissions_map = {}
    print("No permissions CSV found; continuing without.")

try:
    schedules_map = load_schedules_from_csv(exported_path) if apply_schedules else {}
    print(f"Loaded schedules for {len(schedules_map)} dashboard(s)")
except FileNotFoundError:
    schedules_map = {}
    print("No schedules CSV found; continuing without.")

## Build deployment packages

Build deployment packages from transformed JSONs and the loaded permissions/schedules.

In [ ]:
packages = build_deployment_packages(
    transformed_path,
    permissions_map=permissions_map if apply_permissions else None,
    schedules_map=schedules_map if apply_schedules else None
)
print(f"Built {len(packages)} deployment package(s).")

## Deploy using local WorkspaceClient

Use the default WorkspaceClient (this workspace, job identity). No target URL or PAT/SP; no IP check.

In [ ]:
client = WorkspaceClient()
print(f"Deploying to this workspace: {client.config.host}")

result = deploy_via_sdk(
    client,
    packages,
    target_parent_path=target_parent_path,
    warehouse_id=warehouse_id,
    warehouse_name=warehouse_name,
    apply_permissions=apply_permissions,
    apply_schedules=apply_schedules,
    embed_credentials=embed_credentials,
    skip_duplicate_check=skip_duplicate_check,
    dry_run=dry_run_mode
)

print(f"Success: {result.get('successful', 0)}, Failed: {result.get('failed', 0)}, Skipped: {result.get('skipped', 0)}")